In [18]:
# 1. Montar Drive (Para guardar modelos y logs y que no se pierdan)
from google.colab import drive
drive.mount('/content/drive')

# 2. Instalar Librerías
!pip install gymnasium stable-baselines3 sb3-contrib shimmy

# 3. Configurar Path para que Python encuentre tu carpeta 'mus_gym'
import sys
import os

# Ajusta esta ruta a donde subiste tu carpeta en Drive
PROJECT_PATH = '/content/drive/MyDrive/MusProject'
sys.path.append(PROJECT_PATH)

# Verificar que carga
try:
    import mus_gym
    print("✅ Librería mus_gym cargada correctamente desde Drive.")
except ImportError:
    print("❌ Error: No se encuentra mus_gym. Revisa la ruta PROJECT_PATH.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Librería mus_gym cargada correctamente desde Drive.


In [19]:
import gymnasium as gym
import numpy as np
import os
import random
from datetime import datetime
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from sb3_contrib import RecurrentPPO

# Importamos tu entorno
import mus_gym
from mus_gym.mus_env import MusEnvironment
from mus_gym.mus_bots import HeuristicBot
from mus_gym.mus_enums import MusAction

In [21]:

# --- CONFIGURACIÓN COLAB PRO+ ---
# Rutas en Drive para persistencia
DRIVE_ROOT = "/content/drive/MyDrive/MusProject"
MODELS_DIR = os.path.join(DRIVE_ROOT, "models_colab_v4")
LOGS_DIR = os.path.join(DRIVE_ROOT, "logs_colab_v4")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

In [25]:
# Hiperparámetros
NUM_ENVS = 16  # Colab Pro+ suele tener 8-12 vCPUs. 16 procesos satura bien.
TOTAL_TIMESTEPS = 8_000_000 # 3 Millones de pasos totales
PHASE_1_STEPS = 1000000     # Fase de disciplina (vs Sheriff)
CHECKPOINT_FREQ = 250_000
# Si se corta, pon aquí la ruta del zip y pon LOAD_MODEL = True
LOAD_MODEL = True
MODEL_PATH_TO_RESUME = os.path.join(MODELS_DIR, "latest_model_colab")

In [23]:
# --- EL "SHERIFF BOT" (Para castigar a los locos) ---
class SheriffBot(HeuristicBot):
    """
    Un bot que NO se retira si tiene una mano decente.
    Su único propósito es castigar los Órdagos aleatorios de la IA.
    """
    def __init__(self, player_id=1):
        super().__init__(player_id=player_id, prob_farol=0.05, prob_error=0.02)

    def choose_action(self, observation):
        legal = observation["legal_actions_enums"]
        if not legal: return None

        state = observation["game_state"]
        hand_info = observation["my_hand_info"]
        lance = state["current_lance"]

        # Calcular fuerza bruta (0.0 a 1.0)
        fuerza = self._calcular_fuerza(lance, hand_info)

        # Lógica anti-locos:
        if state["bet"] >= 30: # Nos han tirado órdago
            # Si tengo más del 40% de fuerza (mano mediocre), LO ACEPTO.
            # Esto enseña a la IA que no puede tirar órdago con basura.
            if fuerza > 0.40 and MusAction.QUIERO in legal:
                return MusAction.QUIERO
            if MusAction.NO_QUIERO in legal:
                return MusAction.NO_QUIERO

        # Comportamiento normal heredado para el resto
        return super().choose_action(observation)

# --- WRAPPER PARA USAR EL BOT EN GYM ---
class SheriffEnv(MusEnvironment):
    def __init__(self):
        super().__init__(target_score=30)
        self.bot = SheriffBot(player_id=1)
        self.oponente_actual = self.bot # Compatibilidad

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        # Si el bot es mano, juega
        while self.current_player_idx == 1 and not self.game_over:
            obs_dict = self._get_obs_dict_for_bot(1)
            act = self.bot.choose_action(obs_dict)
            if act is None: break
            obs, r, term, trunc, info = super().step(act.value)
            if term or trunc: break
        return obs, info

    def step(self, action):
        o, r, d, t, i = super().step(action)
        if d or t: return o, r, d, t, i

        # Turno del Sheriff
        reward_accum = 0
        while self.current_player_idx == 1 and not (d or t):
            obs_dict = self._get_obs_dict_for_bot(1)
            act = self.bot.choose_action(obs_dict)
            if act is None: break
            o, r_bot, d, t, i = super().step(act.value)
            reward_accum -= r_bot # Zero sum

        return o, r + reward_accum, d, t, i

    def _get_obs_dict_for_bot(self, p_idx):
        # (Copiado de tu entorno_v2 para que funcione inline)
        # Necesitas asegurarte que esto existe en tu clase base o copiarlo aquí
        # Por brevedad, asumo que MusEnvironment tiene acceso a esto o lo heredas de MusEnvConBots
        # Si falla, simplemente usa tu clase MusEnvConBots pero pasándole el Sheriff en __init__
        return super()._get_obs_dict_for_bot(p_idx) if hasattr(super(), '_get_obs_dict_for_bot') else {}

# Para simplificar, usaremos tu clase MusEnvConBots y le inyectaremos el Sheriff
from mus_gym.mus_env_v2 import MusEnvConBots
class SheriffEnvWrapper(MusEnvConBots):
    def __init__(self):
        super().__init__()
        self.bot_oponente = SheriffBot(player_id=1)
        self.oponente_actual = self.bot_oponente

# --- CALLBACK PARA CAMBIAR DE FASE ---
class PhaseChangeCallback(BaseCallback):
    def __init__(self, switch_step, save_path, verbose=1):
        super().__init__(verbose)
        self.switch_step = switch_step
        self.save_path = save_path
        self.switched = False

    def _on_step(self) -> bool:
        if not self.switched and self.num_timesteps >= self.switch_step:
            print(f"\n\n--- 🚨 FASE 1 COMPLETADA ({self.num_timesteps} pasos) ---")
            print("--- Guardando modelo 'Disciplinado' y cambiando estrategia... ---")
            self.model.save(f"{self.save_path}/model_phase1_disciplined")

            # Aquí la magia: NO cambiamos el entorno en caliente (es complejo con SubprocVecEnv).
            # Simplemente guardamos y paramos el entrenamiento para reiniciar la Fase 2 manual o automáticamente.
            # Pero para hacerlo automático, podríamos simplemente dejar que termine y lanzar otro learn.
            self.switched = True
            return False # Detiene el entrenamiento actual
        return True



In [26]:
def main():
    print(f"--- 🚀 INICIANDO ENTRENAMIENTO EN COLAB PRO+ ---")
    print(f"--- Guardando en: {MODELS_DIR} ---")

    # 1. FASE DE DISCIPLINA (vs Sheriff)
    if not LOAD_MODEL:
        print("\n--- [FASE 1] ENTRENANDO CONTRA SHERIFF BOT (DISCIPLINA) ---")
        # Usamos 16 entornos para ir volando
        env_phase1 = make_vec_env(SheriffEnvWrapper, n_envs=NUM_ENVS, vec_env_cls=SubprocVecEnv)

        model = RecurrentPPO(
            "MlpLstmPolicy",
            env_phase1,
            verbose=1,
            tensorboard_log=LOGS_DIR,
            learning_rate=0.0003,
            n_steps=512, # Más pasos por update para estabilizar LSTM
            batch_size=512,
            gamma=0.995,
            ent_coef=0.03, # Entropía media para explorar pero no locura
            policy_kwargs=dict(lstm_hidden_size=256, net_arch=dict(pi=[256, 256], vf=[256, 256]))
        )

        # Guardar cada 100k pasos en Drive
        checkpoint_cb = CheckpointCallback(save_freq=CHECKPOINT_FREQ // NUM_ENVS, save_path=MODELS_DIR, name_prefix="phase1_sheriff")
        phase_cb = PhaseChangeCallback(PHASE_1_STEPS, MODELS_DIR)

        model.learn(total_timesteps=PHASE_1_STEPS + 50000, callback=[checkpoint_cb, phase_cb])
        env_phase1.close()

        # Cargar el modelo resultante para la fase 2
        path_phase1 = f"{MODELS_DIR}/model_phase1_disciplined.zip"
        print(f"Modelo Fase 1 guardado en {path_phase1}")
        model = RecurrentPPO.load(path_phase1)
    else:
        print(f"Reanudando desde modelo: {MODEL_PATH_TO_RESUME}")
        model = RecurrentPPO.load(MODEL_PATH_TO_RESUME)

    # 2. FASE DE SELF-PLAY (LEAGUE)
    # Ahora que ya no somos unos locos, jugamos contra nosotros mismos
    print("\n--- [FASE 2] ENTRENAMIENTO SELF-PLAY (LEAGUE) ---")

    # Preparamos el entorno de SelfPlay
    # Necesitamos crear un entorno que acepte 'opponent_path'
    from mus_gym.mus_selfplay_env import MusEnvSelfPlay

    # Usamos el propio modelo actual como primer oponente
    initial_opponent_path = os.path.join(MODELS_DIR, "opponent_initial.zip")
    model.save(initial_opponent_path)

    env_kwargs = {"opponent_path": initial_opponent_path}
    env_selfplay = make_vec_env(MusEnvSelfPlay, n_envs=NUM_ENVS, vec_env_cls=SubprocVecEnv, env_kwargs=env_kwargs)

    # Re-adjuntamos el entorno al modelo
    model.set_env(env_selfplay)

    # Callback de Liga (Copiado y adaptado para rutas Drive)
    # Lo definimos inline para asegurar que funcione
    class DriveLeagueCallback(BaseCallback):
        def __init__(self, update_freq=50000, verbose=1):
            super().__init__(verbose)
            self.update_freq = update_freq
            self.last_update = 0

        def _on_step(self):
            if self.num_timesteps - self.last_update >= self.update_freq:
                self.last_update = self.num_timesteps
                # 1. Guardar modelo actual como nuevo campeón (sin evaluación para ir rápido en Colab)
                #    En una versión pro, aquí haríamos evaluate_policy.
                #    Para simplificar y evitar timeouts, asumimos que mejora y actualizamos.
                timestamp = datetime.now().strftime("%H%M")
                new_champ_path = os.path.join(MODELS_DIR, f"champion_{timestamp}.zip")

                print(f"[League] 🔄 Actualizando oponente en paso {self.num_timesteps}")
                self.model.save(new_champ_path)

                # Actualizar los entornos
                self.training_env.env_method("update_opponent_model", new_champ_path)

                # Guardar checkpoint de seguridad
                self.model.save(os.path.join(MODELS_DIR, "latest_model_colab"))
            return True

    model.learn(
        total_timesteps=TOTAL_TIMESTEPS - PHASE_1_STEPS,
        callback=DriveLeagueCallback(update_freq=100_000), # Actualizar rival cada 100k pasos
        tb_log_name="PPO_SelfPlay_Colab"
    )

    final_path = os.path.join(MODELS_DIR, "FINAL_MUS_MODEL.zip")
    model.save(final_path)
    print(f"✅ ENTRENAMIENTO COMPLETADO. Modelo final en: {final_path}")
    env_selfplay.close()

if __name__ == "__main__":
    main()

Se han truncado las últimas 5000 líneas del flujo de salida.
|    value_loss           | 0.355      |
----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 11.1       |
|    ep_rew_mean          | 0.494      |
| time/                   |            |
|    fps                  | 357        |
|    iterations           | 619        |
|    time_elapsed         | 14176      |
|    total_timesteps      | 5070848    |
| train/                  |            |
|    approx_kl            | 0.13742083 |
|    clip_fraction        | 0.247      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.383     |
|    explained_variance   | -0.0622    |
|    learning_rate        | 0.0003     |
|    loss                 | 0.0982     |
|    n_updates            | 10700      |
|    policy_gradient_loss | -0.0469    |
|    value_loss           | 0.366      |
-------------------------------------

In [14]:
# --- SCRIPT DE TERAPIA DE CHOQUE (REFORMATORIO) ---
import gymnasium as gym
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
import os
from mus_gym.mus_bots import HeuristicBot
from mus_gym.mus_enums import MusAction
from mus_gym.mus_env_v2 import MusEnvConBots

# RUTAS
DRIVE_ROOT = "/content/drive/MyDrive/MusProject"
MODELS_DIR = os.path.join(DRIVE_ROOT, "models_colab_v4")
LATEST_MODEL = os.path.join(MODELS_DIR, "latest_model_colab.zip")

# --- SUPER SHERIFF (Más estricto) ---
class SuperSheriffBot(HeuristicBot):
    def __init__(self, player_id=1):
        super().__init__(player_id=player_id, prob_farol=0.0, prob_error=0.0) # Cero errores, máquina de matar

    def choose_action(self, observation):
        legal = observation["legal_actions_enums"]
        if not legal: return None

        state = observation["game_state"]
        hand_info = observation["my_hand_info"]
        lance = state["current_lance"]
        fuerza = self._calcular_fuerza(lance, hand_info)

        # SI NOS TIRAN ÓRDAGO
        if state["bet"] >= 30:
            # Antes era 0.40, ahora 0.25.
            # Significa: "Si tengo una mano que no es basura absoluta (25%), TE LO VEO."
            # Esto castigará el 75% de los faroles de tu IA.
            if fuerza > 0.25 and MusAction.QUIERO in legal:
                return MusAction.QUIERO
            if MusAction.NO_QUIERO in legal:
                return MusAction.NO_QUIERO

        return super().choose_action(observation)

# Wrapper para usar el SuperSheriff
class ReformSchoolEnv(MusEnvConBots):
    def __init__(self):
        super().__init__()
        self.bot_oponente = SuperSheriffBot(player_id=1)
        self.oponente_actual = self.bot_oponente

def run_correction():
    print("--- 👮 INICIANDO TERAPIA DE CHOQUE (REFORMATORIO) ---")
    print(f"Cargando al delincuente desde: {LATEST_MODEL}")

    # 1. Creamos el entorno hostil
    # Usamos 16 hilos para que aprenda la lección rápido
    env = make_vec_env(ReformSchoolEnv, n_envs=16, vec_env_cls=SubprocVecEnv)

    # 2. Cargamos el modelo loco
    try:
        model = RecurrentPPO.load(LATEST_MODEL, env=env)
    except Exception as e:
        print(f"Error cargando modelo: {e}")
        return

    # 3. Entrenamos 1.5 Millones de pasos solo contra el Sheriff
    # Bajamos un poco la entropía para forzarle a dejar de probar cosas raras
    model.ent_coef = 0.01

    print("Entrenando 1,000,000 pasos contra Super Sheriff...")
    model.learn(total_timesteps=1_800_000, tb_log_name="PPO_Correction_Therapy")

    # 4. Guardamos el modelo rehabilitado
    rehabilitated_path = os.path.join(MODELS_DIR, "latest_model_colab.zip") # Sobreescribimos el latest
    model.save(rehabilitated_path)
    print(f"✅ REHABILITACIÓN COMPLETADA. Modelo guardado en: {rehabilitated_path}")
    print("Ahora puedes volver a ejecutar el script de Self-Play (Fase 2).")
    env.close()

if __name__ == "__main__":
    run_correction()

--- 👮 INICIANDO TERAPIA DE CHOQUE (REFORMATORIO) ---
Cargando al delincuente desde: /content/drive/MyDrive/MusProject/models_colab_v4/latest_model_colab.zip
Entrenando 1,000,000 pasos contra Super Sheriff...
Logging to /content/drive/MyDrive/MusProject/logs_colab_v4/PPO_Correction_Therapy_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 6        |
|    ep_rew_mean     | 0.877    |
| time/              |          |
|    fps             | 4235     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 8192     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 6.01       |
|    ep_rew_mean          | 0.977      |
| time/                   |            |
|    fps                  | 942        |
|    iterations           | 2          |
|    time_elapsed         | 17         |
|    total_timesteps      | 16384      |

KeyboardInterrupt: 